In [ ]:
"""
Makes a catalogue of morphological data for my galaxies. Creates fits data file Galaxy_morphology.
"""

from astropy.io import fits
from astropy.table import Table
from astropy.table import vstack
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import os
import corner
import asdf
from tqdm import tqdm

with fits.open("/nvme/scratch/work/alberttg/Summer_project/Ha_and_NII_broad_line_data.fits") as hdul:
        data = hdul[1].data
TABLE = Table(data)


GALAXY_ID = TABLE["SURVEY_ID"]        # object ID, used for labeling/output files
SERSIC_FILTERS = ["F444W","F356W","F277W"]                    # filter name, used for labeling/output files
SURVEY = TABLE["SURVEY"]

FLUX_F277W = TABLE["FLUX_F277W"]
FLUX_F356W = TABLE["FLUX_F356W"]
FLUX_F444W = TABLE["FLUX_F444W"]


In [ ]:
def read_summary_table(path, parameter):
    """
    Extracts the mean and sd values for a given parameter from the sersic summary tables.
    """
    csv_table = pd.read_csv(path, index_col=0)
    
    row = csv_table.loc[parameter]
    return row["mean"], row["sd"]

In [ ]:
def read_residual_fits_table(path):
    """
    Opens data_model_residual fits files and returns data for sersic model and residual.
    """
    with fits.open(path) as hdul:
        # sci_im = hdul[1].data
        sersic_model = hdul[2].data
        residual = hdul[3].data
        # mask = hdul[4].data
        rms = hdul[5].data
        print(rms)

    return sersic_model, residual, rms

read_residual_fits_table(f"/nvme/scratch/work/alberttg/Summer_project/Single_sersic_fits/{GALAXY_ID}/{GALAXY_ID}_{filt}_data_model_residual.fits")

Filename: /nvme/scratch/work/alberttg/Summer_project/Single_sersic_fits/942/942_F444W_data_model_residual.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       7   ()      
  1  SCI           1 ImageHDU         8   (32, 32)   float32   
  2  MODEL         1 ImageHDU         9   (32, 32)   float32   
  3  RESIDUAL      1 ImageHDU         9   (32, 32)   float32   
  4  MASK          1 ImageHDU         8   (32, 32)   uint8   
  5  RMS           1 ImageHDU         8   (32, 32)   float32   


In [ ]:
def calculate_RFF(residual, rms, flux_auto):
    """
    Formula for RFF from EPOCHS XI eq 4.
    """

    rff = ( np.sum(np.abs(residual)) - 0.8 * np.sum(np.abs(rms)) ) / flux_auto

    return rff

In [ ]:
def make_single_sersic_table():
    """
    Makes a FITS table of morphological parameters for each galaxy.
    Each galaxy occupies a single row, with separate columns for each filter.
    """

    rows = []

    for i in tqdm(range(len(GALAXY_ID)), total=len(GALAXY_ID)):

        # Start the row with the galaxy information
        row = {
            "SURVEY_ID": GALAXY_ID[i],
            "SURVEY": SURVEY[i],
            "REDSHIFT": TABLE["REDSHIFT"][i],
        }

        # Add morphology measurements for each filter
        for filt in SERSIC_FILTERS:

            summary_path = Path(
                f"/nvme/scratch/work/alberttg/Summer_project/Single_sersic_fits/"
                f"{GALAXY_ID[i]}/{GALAXY_ID[i]}_{filt}_summary.csv"
            )

            if csv_path.exists():
                ellip_mean, ellip_sd = read_summary_table(summary_path, "ellip")
                n_mean, n_sd = read_summary_table(summary_path, "n")
                r_eff_mean, r_eff_sd = read_summary_table(summary_path, "r_eff")
            else:
                ellip_mean = ellip_sd = np.nan
                n_mean = n_sd = np.nan
                r_eff_mean = r_eff_sd = np.nan

            row[f"{filt}_ellip_mean"] = ellip_mean
            row[f"{filt}_ellip_sd"] = ellip_sd
            row[f"{filt}_n_mean"] = n_mean
            row[f"{filt}_n_sd"] = n_sd
            row[f"{filt}_r_eff_mean"] = r_eff_mean
            row[f"{filt}_r_eff_sd"] = r_eff_sd

        # Append one completed row per galaxy
        rows.append(row)

    new_table = Table(rows=rows)

    print(new_table.colnames)

    new_table.write("Galaxy_morphology.fits", format="fits", overwrite=True)

    return new_table

In [4]:
if __name__ == "__main__":
    make_morphology_table()

100%|██████████| 144/144 [00:03<00:00, 40.85it/s]


['SURVEY_ID', 'SURVEY', 'REDSHIFT', 'F444W_ellip_mean', 'F444W_ellip_sd', 'F444W_n_mean', 'F444W_n_sd', 'F444W_r_eff_mean', 'F444W_r_eff_sd', 'F356W_ellip_mean', 'F356W_ellip_sd', 'F356W_n_mean', 'F356W_n_sd', 'F356W_r_eff_mean', 'F356W_r_eff_sd', 'F277W_ellip_mean', 'F277W_ellip_sd', 'F277W_n_mean', 'F277W_n_sd', 'F277W_r_eff_mean', 'F277W_r_eff_sd']
